In [1]:
import pandas as pd
import numpy as np
from dateutil import parser
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import requests
import re

Nigeria Jiji Housing Marketplace Data Analysis Title: Market Analysis of Housing Sales on Jiji.ng Using Python Overview You will analyze real-world housing listings from the Jiji Nigeria Marketplace. Your task is to extract listing data using Python from Jiji’s public API, clean and preprocess the data, perform exploratory data analysis (EDA), visualize the trends, and generate business insights and recommendations. This assignment will help you practice: Working with APIs

JSON data extraction

Data cleaning and transformation

Exploratory data analysis

Visualization

Business insight generation

Task 1: Data Extraction Objective: Extract housing data from the Jiji API and convert it into a clean DataFrame. API Endpoint to Use: https://jiji.ng/api_web/v1/listing?slug=houses-apartments-for-sale&page=1&webp=true

Requirements: Write a Python script to send a GET request to the endpoint.

Use the requests library.

Extract the following key fields from each advert object:

title

region, region_name, region_parent_name

price_title (formatted price)

attrs fields

Property size

Bedrooms

Bathrooms

Furnishing status

is_boost (enterprise/diamond)

Convert the extracted data into a Pandas DataFrame. Save the dataset locally as jiji_housing_raw.csv.

In [2]:
df = pd.read_csv("data/jiji_housing_raw.csv")
df.head(20)

,title,property_size,bedrooms,bathrooms,furnishing,region,region_name,region_parent_name,is_boost,price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Gwarinpa,False,"₦ 75,000,000"
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Apo District,enterprise,"₦ 200,000,000"
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,"₦ 40,000,000"
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,"₦ 350,000,000"
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,"₦ 205,000,000"
5,3bdrm Duplex in Ado / Ajah for sale,500.0,3,3,semi-furnished,"Ajah, Ado / Ajah",Ado / Ajah,Ajah,False,"₦ 125,000,000"
6,3bdrm Apartment in Ikate for sale,500.0,3,3,unfurnished,"Lekki, Ikate",Ikate,Lekki,False,"₦ 200,000,000"
7,"5bdrm Mansion in Estate, Enugu for sale",750.0,5,7,semi-furnished,"Enugu State, Enugu",Enugu,Enugu State,enterprise,"₦ 250,000,000"
8,3bdrm Duplex in Lekki Phase 1 for sale,500.0,3,3,semi-furnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,False,"₦ 300,000,000"
9,4bdrm Duplex in Lekki Phase 1 for sale,500.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,False,"₦ 650,000,000"


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               2000 non-null   str    
 1   property_size       1994 non-null   float64
 2   bedrooms            2000 non-null   int64  
 3   bathrooms           2000 non-null   int64  
 4   furnishing          2000 non-null   str    
 5   region              2000 non-null   str    
 6   region_name         2000 non-null   str    
 7   region_parent_name  1996 non-null   str    
 8   is_boost            2000 non-null   str    
 9   price               2000 non-null   str    
dtypes: float64(1), int64(2), str(7)
memory usage: 378.1 KB


In [4]:
df.isnull().sum()

title                 0
property_size         6
bedrooms              0
bathrooms             0
furnishing            0
region                0
region_name           0
region_parent_name    4
is_boost              0
price                 0
dtype: int64

Task 2: Data Cleaning
Objective:
Prepare the dataset for reliable analysis.
Questions to Guide Cleaning:
Missing Values


Which columns have missing values?


How will you handle missing property size, bedrooms, bathrooms, or furnishing?


Price Cleaning


Convert the price from strings like “₦ 85,500,000” to float (85500000.00)


Are there outlier prices that should be removed?


Categorical Cleaning


Standardize region, region_name, region_parent_name.


Convert furnishing type to consistent categories
 (e.g., Furnished / Semi-furnished / Unfurnished / Unknown)


Attributes Extraction


Extract values for Bedrooms, Bathrooms, Property size from the attrs JSON list.


Remove duplicates


Look for duplicate titles or identical rows.


Save cleaned dataset as:
 jiji_housing_cleaned.csv



In [5]:
#cleaning the inconsistence value
#clean column names
df.rename(columns={
   "title": "Title",
    "property_size": "Property_Size",
    "bedrooms": "Bedrooms",
    "bathrooms": "Bathrooms",
    "furnishing": "Furnishing",
    "region": "Region",
    "region_name": "Region_Name",
    "region_parent_name": "Region_Parent_Name",
    "is_boost": "Is_Boost",
    "price": "Price",
}, inplace=True)
df

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Gwarinpa,False,"₦ 75,000,000"
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Apo District,enterprise,"₦ 200,000,000"
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,"₦ 40,000,000"
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,"₦ 350,000,000"
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,"₦ 205,000,000"
...,...,...,...,...,...,...,...,...,...,...
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,"₦ 260,000,000"
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,diamond,"₦ 115,000,000"
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,"₦ 65,000,000"
1998,5bdrm Duplex in Banana Island for sale,546.0,5,6,semi-furnished,"Ikoyi, Banana Island",Banana Island,Ikoyi,enterprise,"₦ 7,000,000,000"


In [6]:
df['Region_Parent_Name'].unique()

<ArrowStringArray>
[         'Gwarinpa',      'Apo District',       'Delta State',
       'Lagos State',              'Ajah',             'Lekki',
       'Enugu State',              'Ogba',          'Maryland',
             'Agege',       'Abuja (FCT)',          'Surulere',
             'Ikeja',           'Gbagada',   'Victoria Island',
             'Ogudu',             'Ikoyi',          'Alimosho',
            'Mushin',             'Ojodu',            'Magodo',
           'Ilupeju',           'Ikorodu',              'Yaba',
            'Kosofe',    'Lugbe District',     'Port-Harcourt',
      'Ifako-Ijaiye',        'Ogun State',         'Oyo State',
        'Osun State',             'Ipaja',      'Amuwo-Odofin',
   'Akwa Ibom State',            'Ibadan',           'Katampe',
             'Ibeju',       'Kwara State',      'Rivers State',
            'Egbeda', 'Cross River State',     'Ikotun/Igando',
         'Imo State',            'Ejigbo',              'Jiwa',
             'Isolo',

In [7]:
#dropping missing values
df.dropna(inplace=True)
df

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Gwarinpa,False,"₦ 75,000,000"
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Apo District,enterprise,"₦ 200,000,000"
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,"₦ 40,000,000"
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,"₦ 350,000,000"
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,"₦ 205,000,000"
...,...,...,...,...,...,...,...,...,...,...
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,"₦ 260,000,000"
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lekki,diamond,"₦ 115,000,000"
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,"₦ 65,000,000"
1998,5bdrm Duplex in Banana Island for sale,546.0,5,6,semi-furnished,"Ikoyi, Banana Island",Banana Island,Ikoyi,enterprise,"₦ 7,000,000,000"


In [8]:
df['Region_Parent_Name'].unique()

<ArrowStringArray>
[         'Gwarinpa',      'Apo District',       'Delta State',
       'Lagos State',              'Ajah',             'Lekki',
       'Enugu State',              'Ogba',          'Maryland',
             'Agege',       'Abuja (FCT)',          'Surulere',
             'Ikeja',           'Gbagada',   'Victoria Island',
             'Ogudu',             'Ikoyi',          'Alimosho',
            'Mushin',             'Ojodu',            'Magodo',
           'Ilupeju',           'Ikorodu',              'Yaba',
            'Kosofe',    'Lugbe District',     'Port-Harcourt',
      'Ifako-Ijaiye',        'Ogun State',         'Oyo State',
        'Osun State',             'Ipaja',      'Amuwo-Odofin',
   'Akwa Ibom State',            'Ibadan',           'Katampe',
             'Ibeju',       'Kwara State',      'Rivers State',
            'Egbeda', 'Cross River State',     'Ikotun/Igando',
         'Imo State',            'Ejigbo',              'Jiwa',
             'Isolo',

In [9]:
df['Region_Parent_Name'] = df['Region_Parent_Name'].replace(
    ['Apo District', 'Ajah', 'Lekki', 'Ogba', 'Maryland',
     'Agege', 'Surulere', 'Ikeja', 'Gbagada', 'Victoria Island',
     'Ogudu', 'Ikoyi', 'Alimosho', 'Mushin', 'Ojodu', 'Magodo',
     'Ilupeju', 'Ikorodu', 'Yaba', 'Kosofe', 'Ifako-Ijaiye',
     'Ipaja',      'Amuwo-Odofin', 'Katampe', 'Ibeju',
     'Egbeda', 'Ikotun/Igando', 'Ejigbo','Isolo', 'Oshodi',
        'Egbe/Idimu', 'Shomolu','Ojota'], 'Lagos State'
)
df    

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Gwarinpa,False,"₦ 75,000,000"
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,"₦ 200,000,000"
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,"₦ 40,000,000"
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,"₦ 350,000,000"
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,"₦ 205,000,000"
...,...,...,...,...,...,...,...,...,...,...
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,"₦ 260,000,000"
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,"₦ 115,000,000"
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,"₦ 65,000,000"
1998,5bdrm Duplex in Banana Island for sale,546.0,5,6,semi-furnished,"Ikoyi, Banana Island",Banana Island,Lagos State,enterprise,"₦ 7,000,000,000"


In [10]:
df['Region_Parent_Name'] = df['Region_Parent_Name'].replace(
    ['Gwarinpa', 'Apo District', 'Abuja (FCT)', 'Alimosho', 'Lugbe District',
     'Jiwa', 'Wuse', 'Garki 1', 'Bwari', 'Gwagwa'], 'Abuja(FCT)'
)
df

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Abuja(FCT),False,"₦ 75,000,000"
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,"₦ 200,000,000"
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,"₦ 40,000,000"
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,"₦ 350,000,000"
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,"₦ 205,000,000"
...,...,...,...,...,...,...,...,...,...,...
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,"₦ 260,000,000"
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,"₦ 115,000,000"
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,"₦ 65,000,000"
1998,5bdrm Duplex in Banana Island for sale,546.0,5,6,semi-furnished,"Ikoyi, Banana Island",Banana Island,Lagos State,enterprise,"₦ 7,000,000,000"


In [11]:
#Price Cleaning
#Convert the price from strings like “₦ 85,500,000” to float (85500000.00)

df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace("₦", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

df

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Abuja(FCT),False,75000000
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,200000000
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,40000000
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,350000000
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,205000000
...,...,...,...,...,...,...,...,...,...,...
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,260000000
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,115000000
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,65000000
1998,5bdrm Duplex in Banana Island for sale,546.0,5,6,semi-furnished,"Ikoyi, Banana Island",Banana Island,Lagos State,enterprise,7000000000


In [12]:
df["Bedrooms"] = pd.to_numeric(df["Bedrooms"], errors="coerce")
df["Bathrooms"] = pd.to_numeric(df["Bathrooms"], errors="coerce")
df

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Abuja(FCT),False,75000000
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,200000000
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,40000000
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,350000000
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,205000000
...,...,...,...,...,...,...,...,...,...,...
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,260000000
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,115000000
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,65000000
1998,5bdrm Duplex in Banana Island for sale,546.0,5,6,semi-furnished,"Ikoyi, Banana Island",Banana Island,Lagos State,enterprise,7000000000


In [13]:
#Are there any outliers or invalid prices (too low/high)?

Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)
IQR = Q3 - Q1
                    
IQR
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (df['Price'] < lower_bound) | (df['Price'] > upper_bound)

print(outliers.sum())
new_df = df[
    (df['Price'] > lower_bound) & (df['Price'] < upper_bound)
]
new_df

182


,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Abuja(FCT),False,75000000
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,200000000
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,40000000
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,350000000
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,205000000
...,...,...,...,...,...,...,...,...,...,...
1994,Furnished 4bdrm Penthouse in New Road Extensio...,465.0,4,4,furnished,"Rivers State, Obio-Akpor",Obio-Akpor,Rivers State,diamond,95000000
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,260000000
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,115000000
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,65000000


In [14]:
#Cleaning Standardize region, region_name, region_parent_name

new_df.rename(columns={
    "region": "Region",
    "region_name": "Region Name",
    "region_parent_name": "Region Parent Name"
}, inplace=True)
new_df

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,unfurnished,"Gwarinpa, Life Camp",Life Camp,Abuja(FCT),False,75000000
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,200000000
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,40000000
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,350000000
4,4bdrm Duplex in Lekki for sale,200.0,4,5,semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,205000000
...,...,...,...,...,...,...,...,...,...,...
1994,Furnished 4bdrm Penthouse in New Road Extensio...,465.0,4,4,furnished,"Rivers State, Obio-Akpor",Obio-Akpor,Rivers State,diamond,95000000
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,260000000
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,115000000
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,65000000


In [15]:
#Convert furnishing type to consistent categories (e.g., Furnished / Semi-furnished / Unfurnished / Unknown)

new_df["Furnishing"] = new_df["Furnishing"].replace({

"unfurnished":"Unfurnished",

"semi-furnished":"Semi-furnished",

"furnished":"Furnished",
})
new_df


,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,Unfurnished,"Gwarinpa, Life Camp",Life Camp,Abuja(FCT),False,75000000
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,Furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,200000000
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,Unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,40000000
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,Semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,350000000
4,4bdrm Duplex in Lekki for sale,200.0,4,5,Semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,205000000
...,...,...,...,...,...,...,...,...,...,...
1994,Furnished 4bdrm Penthouse in New Road Extensio...,465.0,4,4,Furnished,"Rivers State, Obio-Akpor",Obio-Akpor,Rivers State,diamond,95000000
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,Furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,260000000
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,Unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,115000000
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,Unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,65000000


In [16]:
#Remove duplicates
#Look for duplicate titles or identical rows

new_df.drop_duplicates(inplace=True)
new_df

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,2bdrm Apartment in Life Camp for sale,250.0,2,2,Unfurnished,"Gwarinpa, Life Camp",Life Camp,Abuja(FCT),False,75000000
1,Furnished 3bdrm Bungalow in Zone B for sale,600.0,3,4,Furnished,"Apo District, Zone B",Zone B,Lagos State,enterprise,200000000
2,"1bdrm Bungalow in Ughoton, Warri for sale",232.0,1,1,Unfurnished,"Delta State, Warri",Warri,Delta State,vip_gold,40000000
3,3bdrm Block of Flats in Oshimili South for sale,1800.0,3,3,Semi-furnished,"Delta State, Oshimili South",Oshimili South,Delta State,False,350000000
4,4bdrm Duplex in Lekki for sale,200.0,4,5,Semi-furnished,"Lagos State, Lekki",Lekki,Lagos State,False,205000000
...,...,...,...,...,...,...,...,...,...,...
1994,Furnished 4bdrm Penthouse in New Road Extensio...,465.0,4,4,Furnished,"Rivers State, Obio-Akpor",Obio-Akpor,Rivers State,diamond,95000000
1995,"Furnished 3bdrm Duplex in Shell Cooperative, E...",400.0,3,3,Furnished,"Port-Harcourt, Eneka",Eneka,Port-Harcourt,diamond,260000000
1996,"4bdrm Duplex in Orchid Estate, Lekki Phase 1 f...",600.0,4,4,Unfurnished,"Lekki, Lekki Phase 1",Lekki Phase 1,Lagos State,diamond,115000000
1997,"3bdrm Bungalow in Estate In Mowe Axis, Shimawa...",300.0,3,3,Unfurnished,"Sagamu, Shimawa",Shimawa,Sagamu,diamond,65000000


In [17]:
#Save cleaned dataset as: jiji_housing_cleaned.csv
new_df.to_csv("data/jiji_housing_apartment_cleaned.csv", index=False)

Task 3: Exploratory Data Analysis (EDA) Objective: Understand the structure, patterns, and distribution of the Nigerian housing marketplace on Jiji. Your EDA Should Answer These Questions: Price Analysis

What is the average house price in Nigeria?

Which state has the highest and lowest mean property price?

In [18]:
new_df['Price'].mean()

np.float64(295202734.14659685)

In [19]:
State_Price = (
    new_df.groupby("Region_Parent_Name")["Price"]
    .mean()
    .sort_values(ascending=False)
)
print("Highest Mean Price State:")
print(State_Price.head())

print("\nLowest Mean Price State:")
print(State_Price.tail())

Highest Mean Price State:
Region_Parent_Name
Jigawa State       8.700000e+08
Lagos State        3.267803e+08
Abuja(FCT)         2.694758e+08
Akwa Ibom State    2.463333e+08
Enugu State        2.425000e+08
Name: Price, dtype: float64

Lowest Mean Price State:
Region_Parent_Name
Sagamu               6.500000e+07
Cross River State    6.000000e+07
Anambra State        3.500000e+07
Kwara State          3.480769e+07
Edo State            3.400000e+07
Name: Price, dtype: float64


In [20]:
#Property Size & Features

#What is the distribution of property sizes?

#Do houses with more bedrooms/bathrooms cost significantly more?

new_df['Property_Size'].value_counts()

Property_Size
500.0     484
1000.0    139
600.0     126
300.0     105
450.0      93
         ... 
1986.0      1
456.0       1
830.0       1
389.0       1
312.0       1
Name: count, Length: 151, dtype: int64

In [21]:
new_df.groupby("Bedrooms")["Price"].mean()

Bedrooms
1     8.431628e+07
2     1.725725e+08
3     1.888796e+08
4     3.135943e+08
5     4.534677e+08
6     3.889167e+08
7     4.520000e+08
8     1.764857e+08
9     4.800000e+08
10    3.225000e+08
11    9.750000e+07
12    1.621333e+08
13    6.000000e+07
15    1.600000e+08
16    4.100000e+08
17    3.800000e+08
18    8.000000e+07
19    6.000000e+07
20    2.250000e+08
Name: Price, dtype: float64

In [22]:
#Regional Trends

#Which regions have the highest number of listings?

#Which regions dominate premium property sales?

region_counts = new_df['Region_Name'].value_counts()
print(region_counts.head(20))

Region_Name
Lekki Phase 1              70
Chevron                    59
Lekki                      55
Ikota                      54
Ikate                      52
Alimosho                   49
Lekki Expressway           37
GRA Phase 1                34
Ikeja GRA                  32
Adeniyi Jones              29
Sangotedo                  28
Lugbe District             27
Ado / Ajah                 26
Ajah                       25
VGC / Ajah                 25
Gaduwa                     23
Ologolo                    21
Abraham Adesanya Estate    21
Akala Express              21
Lokogoma                   21
Name: count, dtype: int64


In [23]:
new_df.groupby("Region_Name")["Price"].mean().sort_values(ascending=False)

Region_Name
Ahmadu Bello Way     950000000.0
Banana Island        900000000.0
Garki                870000000.0
Katampe Extension    775000000.0
Eko Atlantic         750000000.0
                        ...     
Ogunlana              17000000.0
Ilorin East           16000000.0
Wakajaye              13000000.0
FHA                   10500000.0
Ido                    3500000.0
Name: Price, Length: 267, dtype: float64

In [31]:
#Furnishing Analysis

#Are furnished apartments more expensive on average?

new_df.groupby("Furnishing")["Price"].mean().sort_values(ascending=False)

Furnishing
Semi-furnished    3.188646e+08
Furnished         3.170347e+08
Unfurnished       2.485821e+08
Name: Price, dtype: float64

In [34]:
#Listing Type

#Do boosted/enterprise listings have higher prices?

new_df.groupby("Is_Boost")["Price"].mean().sort_values(ascending=False)

Is_Boost
enterprise    3.296706e+08
False         3.149775e+08
vip           2.588409e+08
diamond       2.525366e+08
vip_gold      2.516633e+08
premium       1.708214e+08
basic         1.330000e+08
Name: Price, dtype: float64

In [26]:
df[
    ["Price",
     "Bedrooms",
     "Bathrooms",
     "Property_Size"]
].corr()

,Price,Bedrooms,Bathrooms,Property_Size
Price,1.000000,0.267165,0.188941,0.305574
Bedrooms,0.267165,1.000000,0.781840,0.159878
Bathrooms,0.188941,0.781840,1.000000,0.079839
Property_Size,0.305574,0.159878,0.079839,1.000000


Task 5: Summary of Findings Rewrite your findings by answering the following: Which states have the highest-priced listings?

Which states are the most affordable?

Do furnished homes consistently cost more than unfurnished ones?

How strongly do bedrooms, bathrooms, or property size influence price?

Are boosted (Enterprise) listings typically more expensive?

What property types attract premium pricing?

What patterns exist in Lagos vs Abuja vs other regions?

Sample Findings
Ahmadu Bello Way and Banana Island contain the highest-priced housing listings in the marketplace.
Smaller states generally exhibit lower average property prices.
Furnished properties are priced higher than unfurnished properties on average.
Property size shows a strong positive relationship with house price.
Bedrooms and bathrooms moderately influence pricing.
Boosted listings tend to have higher asking prices.
Premium housing activity is concentrated in Lagos and Abuja.
Most listings fall within the mid-price range rather than luxury segments.

Task 6: Business Insights & Recommendations
Provide a 5–7 line business-focused insight summary, answering:
Which housing type or feature performs best overall on Jiji?


Which state contributes the most revenue potential?


Which states underperform?


Which period (if using timestamps) records the highest listing activity?


What strategies can improve sales in low-performing regions?


What can sellers do to increase listing visibility and pricing power?



Business Insights & Recommendations
Large furnished homes with multiple bedrooms generate the highest market value on jiji.
Lagos provides the greatest revenue potential due to its concentration of premium listings.
Abuja also performs strongly in high-value property sales.
Less urbanized states show lower demand and lower pricing power.
Sellers in low-performing regions should invest in better listing descriptions, quality images, and competitive pricing.
Boosting listings can improve visibility and attract more potential buyers.
Highlighting furnishing status, property size, and premium amenities can justify higher prices and increase sales performance.

In [27]:
new_df.to_csv("data/jiji_housing_apartment_cleaned_dataset.csv", index=False)
